# End-to-End KTM + Off-Policy Pipeline

This notebook runs: processing -> KTM dataframe -> Gaussian behavior policy -> propensity tuples -> off-policy training -> visualizations.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'offpolicy_ktm_pipeline').exists():
    ROOT = ROOT.parent
if not (ROOT / 'offpolicy_ktm_pipeline').exists():
    raise RuntimeError('Run this notebook from inside the geometric_flow repo.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print('Repo root:', ROOT)

In [ ]:
from offpolicy_ktm_pipeline.src.dataframe_processing import process_pix_dataset, build_ktm_dataframe
from offpolicy_ktm_pipeline.src.gaussian_regression import fit_gaussian_by_order
from offpolicy_ktm_pipeline.src.propensity import build_offpolicy_dataset
from offpolicy_ktm_pipeline.src.estimators_and_training import (
    train_global_policy,
    get_policy_coefficients,
    mc_policy_value,
    optimal_irt_value,
)
from offpolicy_ktm_pipeline.src.visualization import (
    plot_training_history,
    build_policy_curves,
    plot_policy_curves,
)

In [ ]:
pix_input = ROOT / 'data' / 'pix_data.csv'
if not pix_input.exists():
    raise FileNotFoundError(f'Missing input CSV: {pix_input}')

stats = process_pix_dataset(
    input_csv=str(pix_input),
    output_dir=str(ROOT / 'pix_mapping'),
)
stats

In [ ]:
df_ktm = build_ktm_dataframe(
    processed_csv=str(ROOT / 'pix_mapping' / 'pix_processed.csv'),
    sample_users=10000,  # set 0 to use all users
    seed=42,
    fit_intercept=False,
)
df_ktm.head(), df_ktm.shape

In [ ]:
fit_df = fit_gaussian_by_order(
    df_ktm,
    mu_degree=1,
    sigma_degree=2,
    min_obs_per_order=200,
)
fit_path = ROOT / 'pix_mapping' / 'ktm_gaussian_fit_structured.csv'
fit_df.to_csv(fit_path, index=False)
fit_df.head(), fit_df.shape, fit_path

In [ ]:
offpolicy_df = build_offpolicy_dataset(df_ktm, fit_df, sigma_floor=1e-4)
offpolicy_path = ROOT / 'pix_mapping' / 'ktm_offpolicy_structured.csv'
offpolicy_df.to_csv(offpolicy_path, index=False)
offpolicy_df.head(), offpolicy_df.shape, offpolicy_path

In [ ]:
policy, history_df, coeffs = train_global_policy(
    offpolicy_df,
    fit_df=fit_df,
    objective='snips',
    denominator='mixture',
    mu_degree=1,
    sigma_degree=2,
    epochs=2,   # increase for stronger training
    lr=0.005,
    batch_size=8192,
    num_workers=0,
)
history_df.tail()

In [ ]:
beta_mu, beta_sigma = get_policy_coefficients(policy)
theta = offpolicy_df['proficiency'].to_numpy(dtype=float)
delta_min = float(offpolicy_df['difficulties'].min())
delta_max = float(offpolicy_df['difficulties'].max())

mc_val = mc_policy_value(
    theta=theta,
    beta_mu=beta_mu,
    beta_sigma=beta_sigma,
    delta_min=delta_min,
    n_mc=600,
    seed=42,
)
opt_val = optimal_irt_value(theta=theta, delta_min=delta_min, delta_max=delta_max)
print({'mc_policy_value': mc_val, 'irt_optimal_value': opt_val})

In [ ]:
_ = plot_training_history(history_df, title='Training Metrics (SNIPS, mixture denominator)')
plt.show()

last_order = int(fit_df['order_sequence'].max())
brow = fit_df[fit_df['order_sequence'] == last_order].iloc[0]
b_mu = np.array([brow['beta_mu_0'], brow['beta_mu_1']], dtype=float)
b_sigma = np.array([brow['beta_sigma_0'], brow['beta_sigma_1'], brow['beta_sigma_2']], dtype=float)

theta_grid = np.linspace(float(offpolicy_df['proficiency'].min()), float(offpolicy_df['proficiency'].max()), 400)
curves = build_policy_curves(
    theta_grid=theta_grid,
    beta_mu_behavior=b_mu,
    beta_sigma_behavior=b_sigma,
    beta_mu_learned=beta_mu,
    beta_sigma_learned=beta_sigma,
)
d_grid = np.linspace(delta_min, delta_max, 1200)
z = theta_grid[:, None] - d_grid[None, :]
p = 1.0 / (1.0 + np.exp(-z))
exp_reward = p * (d_grid[None, :] - delta_min)
opt_delta_curve = d_grid[np.argmax(exp_reward, axis=1)]

_ = plot_policy_curves(
    theta_grid=theta_grid,
    behavior_mu=curves['behavior_mu'],
    behavior_sigma=curves['behavior_sigma'],
    learned_mu=curves['learned_mu'],
    learned_sigma=curves['learned_sigma'],
    optimal_delta=opt_delta_curve,
    y_min=-4,
    y_max=4,
    title='Behavior vs Learned Policy (with IRT optimal curve)',
)
plt.show()